In [ ]:
# | default_exp features.atac

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()


In [ ]:
# | export
class ATACEvaluator(FeatureEvaluator):
    """Extracts ATAC footprint metrics per feature set."""

    name = "AtacOnTarget"
    source_file = ".ATAC.ontarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            if "label" in cols:
                metrics = [
                    c for c in ["count", "mean_size", "entropy", "z_score"] if c in cols
                ]
                for row in df.to_dict("records"):
                    lbl = str(row["label"]).replace(" ", "_").replace("-", "_")
                    for m in metrics:
                        if pd.notna(row[m]):
                            extracted[f"{lbl}_{m}"] = float(row[m])

            return extracted
        except Exception as e:
            log.exception("extraction_failed", evaluator=self.name, error=str(e))
            return {}

In [ ]:
# | test
evaluator = ATACEvaluator()
df = pd.DataFrame([{"label": "ACC", "entropy": 7.3, "z_score": 1.2}])
res = evaluator.extract(df)
assert res["ACC_z_score"] == 1.2